In [0]:
%sql

SELECT
    COUNT(*) AS total_orders,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(final_amount) AS total_sales,
    MIN(order_date) AS first_order_date,
    MAX(order_date) AS last_order_date
FROM ecommerce.silver.orders;

total_orders,unique_orders,total_sales,first_order_date,last_order_date
100000,100000,5338856525.64,2024-01-01T00:21:12.000Z,2026-09-29T23:53:03.000Z


**Create Gold Daily Sales**

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS ecommerce.gold
COMMENT 'Gold schema containing business-ready e-commerce analytics datasets';

In [0]:
%sql

DESCRIBE SCHEMA EXTENDED ecommerce.gold;

database_description_item,database_description_value
Catalog Name,ecommerce
Namespace Name,gold
Comment,Gold schema containing business-ready e-commerce analytics datasets
Location,
Owner,saireddy12315_outlook.com#ext#@saireddy12315outlook.onmicrosoft.com
Properties,"((unity.catalog.managed.delta.defaults.delta.parquet.format.version,2.12.0), (unity.catalog.managed.delta.defaults.delta.parquet.format.version.afe.internal,2.12.0), (unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_azure_centralus)


In [0]:
%sql

CREATE TABLE ecommerce.gold.daily_sales
USING DELTA
LOCATION 'abfss://gold@stcommercesrc.dfs.core.windows.net/daily_sales/'
AS
SELECT
    TO_DATE(o.order_date) AS sales_date,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COALESCE(SUM(oi.total_items), 0) AS total_items,
    ROUND(SUM(o.total_amount), 2) AS gross_sales,
    ROUND(SUM(o.discount_amount), 2) AS total_discount,
    ROUND(SUM(o.tax_amount), 2) AS total_tax,
    ROUND(SUM(o.final_amount), 2) AS net_sales,
    ROUND(AVG(o.final_amount), 2) AS average_order_value
FROM ecommerce.silver.orders o
LEFT JOIN (
    SELECT
        order_id,
        SUM(quantity) AS total_items
    FROM ecommerce.silver.order_items
    GROUP BY order_id
) oi
    ON o.order_id = oi.order_id
GROUP BY TO_DATE(o.order_date);

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT *
FROM ecommerce.gold.daily_sales
ORDER BY sales_date
LIMIT 20;

sales_date,total_orders,total_items,gross_sales,total_discount,total_tax,net_sales,average_order_value
2024-01-01,94,849,4481738.00,348810.58,743926.92,4876854.34,51881.43
2024-01-02,94,833,4933416.52,505607.04,797005.77,5224815.25,55583.14
2024-01-03,92,844,4490059.35,464983.49,724513.65,4749589.51,51625.97
2024-01-04,96,867,4587743.06,432420.84,747958.00,4903280.22,51075.84
2024-01-05,113,958,5658178.06,509345.42,926789.87,6075622.51,53766.57
2024-01-06,111,973,6200135.30,549403.42,1017131.74,6667863.62,60070.84
2024-01-07,104,958,4834229.15,468447.20,785840.77,5151622.72,49534.83
2024-01-08,101,794,4879288.73,482893.86,791351.15,5187746.02,51363.82
2024-01-09,94,853,4759979.20,473207.93,771618.74,5058390.01,53812.66
2024-01-10,99,939,5213496.45,521388.61,844579.41,5536687.25,55926.13


In [0]:
%sql

SELECT
    COUNT(*) AS number_of_days,
    SUM(total_orders) AS gold_orders,
    ROUND(SUM(gross_sales), 2) AS gold_gross_sales,
    ROUND(SUM(net_sales), 2) AS gold_net_sales
FROM ecommerce.gold.daily_sales;

number_of_days,gold_orders,gold_gross_sales,gold_net_sales
1003,100000,5028730578.98,5338856525.64


In [0]:
%sql

DESCRIBE DETAIL ecommerce.gold.daily_sales;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,2c3fc23e-41fd-4120-9468-190702517ac8,ecommerce.gold.daily_sales,null,abfss://gold@stcommercesrc.dfs.core.windows.net/daily_sales,2026-10-03T16:54:14.882Z,2026-10-03T16:54:19.000Z,List(),List(),1,40340,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql

DESCRIBE HISTORY ecommerce.gold.daily_sales;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-03T16:54:19.000Z,148417918535894,saireddy12315_outlook.com#ext#@saireddy12315outlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(1220552743077213),92b09c12-67c1-477e-a826-9547743bf5e7,1003-165234-qzfm8qms-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1003, numOutputBytes -> 40340)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


**Product Performance**

In [0]:
%sql

SELECT
    oi.product_id,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    SUM(oi.quantity) AS total_units_sold,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS gross_item_sales,
    ROUND(SUM(oi.discount_amount), 2) AS total_discount,
    ROUND(SUM(oi.tax_amount), 2) AS total_tax,
    ROUND(
        SUM(
            (oi.quantity * oi.unit_price)
            - oi.discount_amount
            + oi.tax_amount
        ),
        2
    ) AS net_item_sales
FROM ecommerce.silver.order_items oi
GROUP BY oi.product_id
ORDER BY net_item_sales DESC
LIMIT 20;

product_id,total_orders,total_units_sold,gross_item_sales,total_discount,total_tax,net_item_sales
1599,87,256,22527951.36,2285940.72,3643561.97,23885572.61
41,74,228,19610938.92,1871908.14,3193025.55,20932056.33
561,77,256,19630850.56,2139225.26,3148492.54,20640117.84
959,77,246,18676843.98,1833642.98,3031776.21,19874977.21
38,70,222,18692539.86,1915758.27,3019820.67,19796602.26
4139,71,222,18673874.10,2116025.20,2980412.77,19538261.67
2040,66,220,18349071.40,1908822.92,2959244.72,19399493.20
2670,66,217,18248651.89,1821593.27,2956870.52,19383929.14
4981,67,219,18278937.93,2015584.25,2927403.62,19190757.30
3652,77,227,18068178.50,1841084.06,2920876.96,19147971.40


In [0]:
%sql

CREATE TABLE ecommerce.gold.product_performance
USING DELTA
LOCATION 'abfss://gold@stcommercesrc.dfs.core.windows.net/product_performance/'
AS
SELECT
    oi.product_id,
    COALESCE(p.product_name, 'Unknown Product') AS product_name,
    COALESCE(p.category_name, 'Unknown Category') AS category_name,
    COALESCE(p.brand, 'Unknown Brand') AS brand,

    COUNT(DISTINCT oi.order_id) AS total_orders,
    SUM(oi.quantity) AS total_units_sold,

    ROUND(
        SUM(oi.quantity * oi.unit_price),
        2
    ) AS gross_sales,

    ROUND(
        SUM(oi.discount_amount),
        2
    ) AS total_discount,

    ROUND(
        SUM(oi.tax_amount),
        2
    ) AS total_tax,

    ROUND(
        SUM(
            (oi.quantity * oi.unit_price)
            - oi.discount_amount
            + oi.tax_amount
        ),
        2
    ) AS net_sales,

    ROUND(
        SUM(oi.quantity * oi.unit_price)
        / NULLIF(SUM(oi.quantity), 0),
        2
    ) AS average_selling_price

FROM ecommerce.silver.order_items oi

LEFT JOIN ecommerce.silver.products p
    ON oi.product_id = p.product_id

GROUP BY
    oi.product_id,
    p.product_name,
    p.category_name,
    p.brand;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS product_count,
    SUM(total_orders) AS total_product_orders,
    SUM(total_units_sold) AS total_units_sold,
    ROUND(SUM(gross_sales), 2) AS total_gross_sales,
    ROUND(SUM(total_discount), 2) AS total_discount,
    ROUND(SUM(total_tax), 2) AS total_tax,
    ROUND(SUM(net_sales), 2) AS total_net_sales
FROM ecommerce.gold.product_performance;

product_count,total_product_orders,total_units_sold,total_gross_sales,total_discount,total_tax,total_net_sales
5000,299900,900608,32945091415.62,3285789368.97,5338674355.44,34997976402.09


In [0]:
%sql

SELECT
    ROUND(SUM(quantity * unit_price), 2) AS silver_gross_sales,
    ROUND(SUM(discount_amount), 2) AS silver_discount,
    ROUND(SUM(tax_amount), 2) AS silver_tax,
    ROUND(
        SUM(
            (quantity * unit_price)
            - discount_amount
            + tax_amount
        ),
        2
    ) AS silver_net_sales
FROM ecommerce.silver.order_items;

silver_gross_sales,silver_discount,silver_tax,silver_net_sales
32945091415.62,3285789368.97,5338674355.44,34997976402.09


In [0]:
%sql

SELECT *
FROM ecommerce.gold.product_performance
WHERE product_id = 3998;

product_id,product_name,category_name,brand,total_orders,total_units_sold,gross_sales,total_discount,total_tax,net_sales,average_selling_price
3998,Unknown Product,Unknown Category,Unknown Brand,57,164,8564430.96,855036.41,1387691.01,9097085.56,52222.14


In [0]:
%sql

DESCRIBE DETAIL ecommerce.gold.product_performance;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,2ecfa3b0-4d55-42f1-9503-59d4dc7ed65b,ecommerce.gold.product_performance,null,abfss://gold@stcommercesrc.dfs.core.windows.net/product_performance,2026-10-03T16:59:09.712Z,2026-10-03T16:59:11Z,List(),List(),1,263624,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql

DESCRIBE HISTORY ecommerce.gold.product_performance;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-03T16:59:11Z,148417918535894,saireddy12315_outlook.com#ext#@saireddy12315outlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(1220552743077213),1003-165234-qzfm8qms-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5000, numOutputBytes -> 263624)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


**Customer 360**

In [0]:
%sql

SELECT
    customer_id,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(final_amount), 2) AS total_spend,
    ROUND(AVG(final_amount), 2) AS average_order_value
FROM ecommerce.silver.orders
GROUP BY customer_id
ORDER BY total_spend DESC
LIMIT 20;

customer_id,total_orders,total_spend,average_order_value
5764,13,843229.35,64863.80
1629,13,833774.06,64136.47
4716,10,812034.95,81203.50
24058,11,785593.84,71417.62
15499,11,780100.45,70918.22
12473,9,770505.55,85611.73
13676,9,763266.76,84807.42
5439,10,756760.23,75676.02
7752,12,747702.65,62308.55
128,11,744165.98,67651.45


In [0]:
%sql

SELECT
    o.customer_id,
    SUM(oi.quantity) AS total_items
FROM ecommerce.silver.orders o
INNER JOIN ecommerce.silver.order_items oi
    ON o.order_id = oi.order_id
GROUP BY o.customer_id
ORDER BY total_items DESC
LIMIT 20;

customer_id,total_items
9014,153
20992,150
8203,142
2961,139
13761,138
10053,137
24890,134
9537,132
19858,128
23236,128


In [0]:
%sql

SELECT
    o.customer_id,
    COUNT(DISTINCT CASE
        WHEN UPPER(TRIM(p.payment_status)) = 'SUCCESS'
        THEN p.payment_id
    END) AS successful_payments,

    ROUND(
        SUM(
            CASE
                WHEN UPPER(TRIM(p.payment_status)) = 'SUCCESS'
                THEN p.payment_amount
                ELSE 0
            END
        ),
        2
    ) AS total_paid_amount

FROM ecommerce.silver.orders o

INNER JOIN ecommerce.silver.payments p
    ON o.order_id = p.order_id

GROUP BY o.customer_id
ORDER BY total_paid_amount DESC
LIMIT 20;

customer_id,successful_payments,total_paid_amount
8673,16,1064995.64
20273,15,1060616.90
11247,10,977861.48
20451,15,966213.84
16787,14,962207.63
4993,10,956471.13
16926,13,956221.25
5141,15,955407.90
17642,10,942218.28
17815,13,932837.99


In [0]:
%sql

SELECT
    customer_id,
    COUNT(DISTINCT review_id) AS total_reviews,
    ROUND(AVG(rating), 2) AS average_rating
FROM ecommerce.silver.reviews
GROUP BY customer_id
ORDER BY total_reviews DESC, average_rating DESC
LIMIT 20;

customer_id,total_reviews,average_rating
20964,12,4.42
1388,12,4.33
1861,12,3.67
24267,11,4.64
949,11,4.18
17305,11,4.18
5876,11,3.82
18055,11,3.36
428,10,4.7
2286,10,4.6


In [0]:
%sql

SELECT
    customer_id,
    address_id,
    city,
    state,
    country,
    is_default,
    is_effective_default
FROM ecommerce.silver.customer_addresses
WHERE is_effective_default = true
ORDER BY customer_id
LIMIT 20;

customer_id,address_id,city,state,country,is_default,is_effective_default
2,28656,Gorakhpur,Delhi,India,true,true
4,13048,Kolkata,Rajasthan,India,true,true
12,27561,Latur,Maharashtra,India,true,true
13,23489,Farrukhabad,Rajasthan,India,true,true
15,22820,Panihati,Delhi,India,true,true
17,5865,Pune,Telangana,India,true,true
29,25795,Ghaziabad,Telangana,India,true,true
30,11535,Gangtok,Kerala,India,true,true
32,19930,Ambarnath,Tamil Nadu,India,true,true
33,5036,Aurangabad,Delhi,India,true,true


In [0]:
%sql

CREATE TABLE ecommerce.gold.customer_360
USING DELTA
LOCATION 'abfss://gold@stcommercesrc.dfs.core.windows.net/customer_360/'
AS

WITH order_metrics AS (

    SELECT
        customer_id,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(SUM(final_amount), 2) AS total_spend,
        ROUND(AVG(final_amount), 2) AS average_order_value
    FROM ecommerce.silver.orders
    GROUP BY customer_id

),

item_metrics AS (

    SELECT
        o.customer_id,
        SUM(oi.quantity) AS total_items
    FROM ecommerce.silver.orders o
    INNER JOIN ecommerce.silver.order_items oi
        ON o.order_id = oi.order_id
    GROUP BY o.customer_id

),

payment_metrics AS (

    SELECT
        o.customer_id,

        COUNT(DISTINCT CASE
            WHEN UPPER(TRIM(p.payment_status)) = 'SUCCESS'
            THEN p.payment_id
        END) AS successful_payments,

        ROUND(
            SUM(
                CASE
                    WHEN UPPER(TRIM(p.payment_status)) = 'SUCCESS'
                    THEN p.payment_amount
                    ELSE 0
                END
            ),
            2
        ) AS total_paid_amount

    FROM ecommerce.silver.orders o
    INNER JOIN ecommerce.silver.payments p
        ON o.order_id = p.order_id
    GROUP BY o.customer_id

),

review_metrics AS (

    SELECT
        customer_id,
        COUNT(DISTINCT review_id) AS total_reviews,
        ROUND(AVG(rating), 2) AS average_rating
    FROM ecommerce.silver.reviews
    GROUP BY customer_id

),

address_metrics AS (

    SELECT
        customer_id,
        city,
        state,
        country
    FROM ecommerce.silver.customer_addresses
    WHERE is_effective_default = true

)

SELECT

    c.customer_id,
    c.first_name,
    c.last_name,
    c.email,
    c.gender,
    c.customer_status,

    a.city,
    a.state,
    a.country,

    COALESCE(o.total_orders, 0) AS total_orders,
    COALESCE(i.total_items, 0) AS total_items,

    COALESCE(o.total_spend, 0) AS total_spend,
    COALESCE(o.average_order_value, 0) AS average_order_value,

    COALESCE(p.successful_payments, 0) AS successful_payments,
    COALESCE(p.total_paid_amount, 0) AS total_paid_amount,

    COALESCE(r.total_reviews, 0) AS total_reviews,
    r.average_rating

FROM ecommerce.silver.customers c

LEFT JOIN order_metrics o
    ON c.customer_id = o.customer_id

LEFT JOIN item_metrics i
    ON c.customer_id = i.customer_id

LEFT JOIN payment_metrics p
    ON c.customer_id = p.customer_id

LEFT JOIN review_metrics r
    ON c.customer_id = r.customer_id

LEFT JOIN address_metrics a
    ON c.customer_id = a.customer_id;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS customer_count,
    COUNT(DISTINCT customer_id) AS unique_customer_count
FROM ecommerce.gold.customer_360;

customer_count,unique_customer_count
24997,24997


In [0]:
%sql

SELECT
    customer_id,
    first_name,
    last_name,
    email,
    city,
    state,
    total_orders,
    total_items,
    total_spend,
    average_order_value,
    successful_payments,
    total_paid_amount,
    total_reviews,
    average_rating
FROM ecommerce.gold.customer_360
ORDER BY total_spend DESC
LIMIT 20;

customer_id,first_name,last_name,email,city,state,total_orders,total_items,total_spend,average_order_value,successful_payments,total_paid_amount,total_reviews,average_rating
5764,Vincent,Narang,zilmilbhalla@example.net,Bharatpur,Telangana,13,80,843229.35,64863.80,14,828004.27,3,4.33
1629,Leela,Din,pradhankiaan@example.com,null,null,13,121,833774.06,64136.47,6,386843.68,8,3.88
4716,Oscar,Khurana,nehawalia@example.com,null,null,10,79,812034.95,81203.50,9,771242.09,2,4.0
24058,Yatan,Agarwal,gghosh@example.net,null,null,11,68,785593.84,71417.62,6,373997.42,1,5.0
15499,Manbir,Bhattacharyya,ekeer@example.com,Aurangabad,Telangana,11,68,780100.45,70918.22,12,709981.22,3,4.67
12473,Ojas,Lal,ysankaran@example.com,Salem,Tamil Nadu,9,64,770505.55,85611.73,7,630084.27,3,4.67
13676,Lakshmi,Mangat,hdayal@example.org,Giridih,Tamil Nadu,9,83,763266.76,84807.42,4,396690.22,5,4.8
5439,Anita,Chad,khoslanitara@example.com,null,null,10,78,756760.23,75676.02,1,36256.54,2,5.0
7752,Jackson,Mahal,chatresh61@example.com,Gorakhpur,Andhra Pradesh,12,107,747702.65,62308.55,8,494221.57,5,2.6
128,Isaiah,Balay,pbabu@example.org,null,null,11,88,744165.98,67651.45,11,677432.45,3,4.33


In [0]:
%sql

SELECT
    customer_id,
    COUNT(*) AS record_count
FROM ecommerce.gold.customer_360
GROUP BY customer_id
HAVING COUNT(*) > 1;

customer_id,record_count


In [0]:
%sql

SELECT
    customer_id,
    COUNT(*) AS record_count
FROM ecommerce.gold.customer_360
GROUP BY customer_id
HAVING COUNT(*) > 1;

customer_id,record_count


In [0]:
%sql

SELECT
    ROUND(SUM(final_amount), 2) AS silver_total_spend
FROM ecommerce.silver.orders;

silver_total_spend
5338856525.64


In [0]:
%sql

DESCRIBE DETAIL ecommerce.gold.customer_360;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,7c64fb4b-b5f8-4db2-9553-c14fc5a2f153,ecommerce.gold.customer_360,null,abfss://gold@stcommercesrc.dfs.core.windows.net/customer_360,2026-10-03T17:07:00.674Z,2026-10-03T17:07:03Z,List(),List(),1,979923,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql

DESCRIBE HISTORY ecommerce.gold.customer_360;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-03T17:07:03Z,148417918535894,saireddy12315_outlook.com#ext#@saireddy12315outlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1220552743077213),1003-052435-xepp1xgc,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 24997, numOutputBytes -> 979923)",null,Databricks-Runtime/17.3.x-photon-scala2.13


**Regional Sales**

In [0]:
%sql

SELECT
    COALESCE(a.state, 'Unknown State') AS state,
    COALESCE(a.city, 'Unknown City') AS city,

    COUNT(DISTINCT o.order_id) AS total_orders,

    ROUND(SUM(o.final_amount), 2) AS total_sales

FROM ecommerce.silver.orders o

LEFT JOIN ecommerce.silver.customers c
    ON o.customer_id = c.customer_id

LEFT JOIN ecommerce.silver.customer_addresses a
    ON c.customer_id = a.customer_id
    AND a.is_effective_default = true

GROUP BY
    COALESCE(a.state, 'Unknown State'),
    COALESCE(a.city, 'Unknown City')

ORDER BY total_sales DESC
LIMIT 20;

state,city,total_orders,total_sales
Unknown State,Unknown City,54104,2887130341.97
Maharashtra,Tirunelveli,60,3425128.39
Rajasthan,Ghaziabad,54,3131161.97
Kerala,Satara,48,2685024.58
Gujarat,Guntakal,46,2548459.25
Gujarat,Dindigul,40,2541385.72
West Bengal,Belgaum,47,2501156.02
Rajasthan,Srinagar,43,2380845.50
Kerala,Tiruchirappalli,37,2368208.89
Karnataka,Malegaon,43,2325727.36


In [0]:
%sql

SELECT
    ROUND(SUM(final_amount), 2) AS silver_total_sales
FROM ecommerce.silver.orders;

silver_total_sales
5338856525.64


In [0]:
%sql

CREATE TABLE ecommerce.gold.regional_sales
USING DELTA
LOCATION 'abfss://gold@stcommercesrc.dfs.core.windows.net/regional_sales/'
AS

SELECT
    COALESCE(a.state, 'Unknown State') AS state,
    COALESCE(a.city, 'Unknown City') AS city,

    COUNT(DISTINCT o.order_id) AS total_orders,

    ROUND(SUM(o.final_amount), 2) AS total_sales,

    ROUND(AVG(o.final_amount), 2) AS average_order_value

FROM ecommerce.silver.orders o

LEFT JOIN ecommerce.silver.customers c
    ON o.customer_id = c.customer_id

LEFT JOIN ecommerce.silver.customer_addresses a
    ON c.customer_id = a.customer_id
    AND a.is_effective_default = true

GROUP BY
    COALESCE(a.state, 'Unknown State'),
    COALESCE(a.city, 'Unknown City');

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(DISTINCT o.customer_id) AS customers_with_orders,
    COUNT(DISTINCT CASE
        WHEN a.customer_id IS NOT NULL THEN o.customer_id
    END) AS customers_with_default_address,
    COUNT(DISTINCT CASE
        WHEN a.customer_id IS NULL THEN o.customer_id
    END) AS customers_without_default_address
FROM ecommerce.silver.orders o
LEFT JOIN ecommerce.silver.customer_addresses a
    ON o.customer_id = a.customer_id
    AND a.is_effective_default = true;

customers_with_orders,customers_with_default_address,customers_without_default_address
24568,11069,13499


In [0]:
%sql

SELECT
    COUNT(*) AS affected_order_rows,
    COUNT(DISTINCT o.order_id) AS affected_orders,
    ROUND(SUM(o.final_amount), 2) AS affected_sales
FROM ecommerce.silver.orders o
LEFT JOIN ecommerce.silver.customer_addresses a
    ON o.customer_id = a.customer_id
    AND a.is_effective_default = true
WHERE a.customer_id IS NULL;

affected_order_rows,affected_orders,affected_sales
54096,54096,2886928255.71


In [0]:
%sql

SELECT
    SUM(total_orders) AS regional_orders,
    ROUND(SUM(total_sales), 2) AS regional_sales
FROM ecommerce.gold.regional_sales;

regional_orders,regional_sales
100000,5338856525.64


In [0]:
%sql
SELECT
    customer_id,
    state,
    city,
    is_effective_default
FROM ecommerce.silver.customer_addresses
WHERE
       (state = 'Telangana' AND city IN ('Delhi', 'Tinsukia', 'Yamunanagar'))
    OR (state = 'Kerala' AND city = 'Satara')
    OR (state = 'Rajasthan' AND city = 'Ghaziabad')
    OR (state = 'West Bengal' AND city = 'Jaipur')
LIMIT 20;

customer_id,state,city,is_effective_default
320,Kerala,Satara,true
1119,Telangana,Tinsukia,false
1335,Rajasthan,Ghaziabad,true
1395,Telangana,Tinsukia,true
1548,Rajasthan,Ghaziabad,true
1605,West Bengal,Jaipur,false
2102,Telangana,Yamunanagar,false
2233,Telangana,Tinsukia,false
2305,Telangana,Tinsukia,false
2485,West Bengal,Jaipur,false


In [0]:
%sql

SELECT
    COUNT(*) AS region_count
FROM ecommerce.gold.regional_sales;

region_count
3068


In [0]:
%sql

SELECT
    state,
    city,
    total_orders,
    total_sales,
    average_order_value
FROM ecommerce.gold.regional_sales
ORDER BY total_sales DESC
LIMIT 20;

state,city,total_orders,total_sales,average_order_value
Unknown State,Unknown City,54104,2887130341.97,53362.60
Maharashtra,Tirunelveli,60,3425128.39,57085.47
Rajasthan,Ghaziabad,54,3131161.97,57984.48
Kerala,Satara,48,2685024.58,55938.01
Gujarat,Guntakal,46,2548459.25,55401.29
Gujarat,Dindigul,40,2541385.72,63534.64
West Bengal,Belgaum,47,2501156.02,53216.09
Rajasthan,Srinagar,43,2380845.50,55368.50
Kerala,Tiruchirappalli,37,2368208.89,64005.65
Karnataka,Malegaon,43,2325727.36,54086.68


In [0]:
%sql

SELECT
    ROUND(SUM(total_sales), 2) AS regional_total_sales,
    (
        SELECT ROUND(SUM(final_amount), 2)
        FROM ecommerce.silver.orders
    ) AS silver_total_sales
FROM ecommerce.gold.regional_sales;

regional_total_sales,silver_total_sales
5338856525.64,5338856525.64


**Customer Segmentation**

In [0]:
%sql

SELECT
    c.customer_id,
    c.first_name,
    c.last_name,

    COUNT(DISTINCT o.order_id) AS total_orders,

    ROUND(SUM(o.final_amount), 2) AS total_spend,

    ROUND(AVG(o.final_amount), 2) AS average_order_value,

    MAX(o.order_date) AS last_order_date,

    DATEDIFF(
        CURRENT_DATE(),
        TO_DATE(MAX(o.order_date))
    ) AS recency_days

FROM ecommerce.silver.customers c

LEFT JOIN ecommerce.silver.orders o
    ON c.customer_id = o.customer_id

GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name

ORDER BY total_spend DESC
LIMIT 20;

customer_id,first_name,last_name,total_orders,total_spend,average_order_value,last_order_date,recency_days
5764,Vincent,Narang,13,843229.35,64863.80,2025-12-25T23:29:55Z,282
1629,Leela,Din,13,833774.06,64136.47,2026-09-03T23:26:52Z,30
4716,Oscar,Khurana,10,812034.95,81203.50,2026-04-29T01:07:49Z,157
24058,Yatan,Agarwal,11,785593.84,71417.62,2026-08-13T00:44:07Z,51
15499,Manbir,Bhattacharyya,11,780100.45,70918.22,2026-08-22T17:58:34Z,42
12473,Ojas,Lal,9,770505.55,85611.73,2026-09-11T17:43:03Z,22
13676,Lakshmi,Mangat,9,763266.76,84807.42,2026-08-17T14:14:37Z,47
5439,Anita,Chad,10,756760.23,75676.02,2026-07-15T02:16:10Z,80
7752,Jackson,Mahal,12,747702.65,62308.55,2025-12-16T01:20:54Z,291
128,Isaiah,Balay,11,744165.98,67651.45,2026-06-18T14:46:18Z,107


In [0]:
%sql

WITH customer_metrics AS (

    SELECT
        c.customer_id,
        c.first_name,
        c.last_name,

        COUNT(DISTINCT o.order_id) AS total_orders,

        COALESCE(ROUND(SUM(o.final_amount), 2), 0) AS total_spend,

        COALESCE(ROUND(AVG(o.final_amount), 2), 0) AS average_order_value,

        MAX(o.order_date) AS last_order_date,

        COALESCE(
            DATEDIFF(
                CURRENT_DATE(),
                TO_DATE(MAX(o.order_date))
            ),
            9999
        ) AS recency_days

    FROM ecommerce.silver.customers c

    LEFT JOIN ecommerce.silver.orders o
        ON c.customer_id = o.customer_id

    GROUP BY
        c.customer_id,
        c.first_name,
        c.last_name
),

rfm_scores AS (

    SELECT
        *,

        NTILE(5) OVER (
            ORDER BY recency_days DESC
        ) AS recency_score,

        NTILE(5) OVER (
            ORDER BY total_orders
        ) AS frequency_score,

        NTILE(5) OVER (
            ORDER BY total_spend
        ) AS monetary_score

    FROM customer_metrics
)

SELECT *
FROM rfm_scores
ORDER BY monetary_score DESC,
         frequency_score DESC,
         recency_score DESC
LIMIT 20;

customer_id,first_name,last_name,total_orders,total_spend,average_order_value,last_order_date,recency_days,recency_score,frequency_score,monetary_score
17600,Girish,Bath,9,588241.70,65360.19,2026-08-05T05:25:43Z,59,5,5,5
13243,Theodore,Sathe,8,502973.16,62871.65,2026-08-05T08:27:52Z,59,5,5,5
24786,Ranbir,Saini,7,334642.48,47806.07,2026-08-05T09:58:48Z,59,5,5,5
22921,Utkarsh,Sastry,9,359477.11,39941.90,2026-08-05T14:52:52Z,59,5,5,5
9693,Xavier,Deol,6,387299.70,64549.95,2026-08-05T17:59:56Z,59,5,5,5
11060,Nicholas,Warrior,7,413086.50,59012.36,2026-08-05T11:30:54Z,59,5,5,5
10029,Siddharth,Bali,6,382476.62,63746.10,2026-08-05T06:55:42Z,59,5,5,5
3059,Tanveer,Khalsa,6,328509.65,54751.61,2026-08-05T16:41:36Z,59,5,5,5
16212,Manya,Nayak,6,321103.22,53517.20,2026-08-06T04:59:04Z,58,5,5,5
18816,Sara,Sawhney,7,359906.24,51415.18,2026-08-05T07:25:06Z,59,5,5,5


In [0]:
%sql

WITH customer_metrics AS (

    SELECT
        c.customer_id,
        c.first_name,
        c.last_name,

        COUNT(DISTINCT o.order_id) AS total_orders,

        COALESCE(
            ROUND(SUM(o.final_amount), 2),
            0
        ) AS total_spend,

        COALESCE(
            ROUND(AVG(o.final_amount), 2),
            0
        ) AS average_order_value,

        MAX(o.order_date) AS last_order_date,

        COALESCE(
            DATEDIFF(
                CURRENT_DATE(),
                TO_DATE(MAX(o.order_date))
            ),
            9999
        ) AS recency_days

    FROM ecommerce.silver.customers c

    LEFT JOIN ecommerce.silver.orders o
        ON c.customer_id = o.customer_id

    GROUP BY
        c.customer_id,
        c.first_name,
        c.last_name
),

rfm_scores AS (

    SELECT
        *,

        NTILE(5) OVER (
            ORDER BY recency_days DESC
        ) AS recency_score,

        NTILE(5) OVER (
            ORDER BY total_orders
        ) AS frequency_score,

        NTILE(5) OVER (
            ORDER BY total_spend
        ) AS monetary_score

    FROM customer_metrics
),

segmented_customers AS (

    SELECT
        *,

        CASE

            WHEN recency_score >= 4
             AND frequency_score >= 4
             AND monetary_score >= 4
            THEN 'Champions'

            WHEN frequency_score >= 4
             AND recency_score >= 3
            THEN 'Loyal Customers'

            WHEN monetary_score >= 4
            THEN 'Big Spenders'

            WHEN recency_score >= 4
             AND frequency_score <= 3
            THEN 'Recent Customers'

            WHEN recency_score <= 2
             AND frequency_score >= 3
            THEN 'At Risk'

            WHEN recency_score = 3
            THEN 'Needs Attention'

            ELSE 'Lost Customers'

        END AS customer_segment

    FROM rfm_scores
)

SELECT
    customer_segment,
    COUNT(*) AS customer_count,

    ROUND(
        SUM(total_spend),
        2
    ) AS segment_total_spend,

    ROUND(
        AVG(total_spend),
        2
    ) AS average_customer_spend,

    ROUND(
        AVG(total_orders),
        2
    ) AS average_orders

FROM segmented_customers

GROUP BY customer_segment

ORDER BY customer_count DESC;

customer_segment,customer_count,segment_total_spend,average_customer_spend,average_orders
Lost Customers,5748,567491205.79,98728.46,1.91
Champions,4496,1606471525.89,357311.28,6.34
Big Spenders,3724,1121688914.83,301205.40,4.7
Recent Customers,3686,523618946.90,142056.14,2.94
Loyal Customers,3250,894919444.31,275359.83,5.64
Needs Attention,2295,312878395.60,136330.46,2.83
At Risk,1798,311293461.62,173133.18,4.08


In [0]:
%sql

CREATE TABLE ecommerce.gold.customer_segmentation
USING DELTA
LOCATION 'abfss://gold@stcommercesrc.dfs.core.windows.net/customer_segmentation/'
AS

WITH customer_metrics AS (

    SELECT
        c.customer_id,
        c.first_name,
        c.last_name,
        c.email,
        c.gender,
        c.customer_status,

        COUNT(DISTINCT o.order_id) AS total_orders,

        COALESCE(
            ROUND(SUM(o.final_amount), 2),
            0
        ) AS total_spend,

        COALESCE(
            ROUND(AVG(o.final_amount), 2),
            0
        ) AS average_order_value,

        MAX(o.order_date) AS last_order_date,

        COALESCE(
            DATEDIFF(
                CURRENT_DATE(),
                TO_DATE(MAX(o.order_date))
            ),
            9999
        ) AS recency_days

    FROM ecommerce.silver.customers c

    LEFT JOIN ecommerce.silver.orders o
        ON c.customer_id = o.customer_id

    GROUP BY
        c.customer_id,
        c.first_name,
        c.last_name,
        c.email,
        c.gender,
        c.customer_status
),

rfm_scores AS (

    SELECT
        *,

        NTILE(5) OVER (
            ORDER BY recency_days DESC
        ) AS recency_score,

        NTILE(5) OVER (
            ORDER BY total_orders
        ) AS frequency_score,

        NTILE(5) OVER (
            ORDER BY total_spend
        ) AS monetary_score

    FROM customer_metrics
),

segmented_customers AS (

    SELECT
        *,

        CASE

            WHEN recency_score >= 4
             AND frequency_score >= 4
             AND monetary_score >= 4
            THEN 'Champions'

            WHEN frequency_score >= 4
             AND recency_score >= 3
            THEN 'Loyal Customers'

            WHEN monetary_score >= 4
            THEN 'Big Spenders'

            WHEN recency_score >= 4
             AND frequency_score <= 3
            THEN 'Recent Customers'

            WHEN recency_score <= 2
             AND frequency_score >= 3
            THEN 'At Risk'

            WHEN recency_score = 3
            THEN 'Needs Attention'

            ELSE 'Lost Customers'

        END AS customer_segment

    FROM rfm_scores
)

SELECT
    customer_id,
    first_name,
    last_name,
    email,
    gender,
    customer_status,

    total_orders,
    total_spend,
    average_order_value,

    last_order_date,
    recency_days,

    recency_score,
    frequency_score,
    monetary_score,

    customer_segment,

    CURRENT_TIMESTAMP() AS segmentation_timestamp

FROM segmented_customers;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS total_customers,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_customer_records
FROM ecommerce.gold.customer_segmentation;

total_customers,unique_customers,duplicate_customer_records
24997,24997,0


In [0]:
%sql

SELECT
    customer_segment,
    COUNT(*) AS customer_count,
    ROUND(SUM(total_spend), 2) AS total_spend,
    ROUND(AVG(total_spend), 2) AS average_spend,
    ROUND(AVG(total_orders), 2) AS average_orders
FROM ecommerce.gold.customer_segmentation
GROUP BY customer_segment
ORDER BY customer_count DESC;

customer_segment,customer_count,total_spend,average_spend,average_orders
Lost Customers,5748,567491205.79,98728.46,1.91
Champions,4496,1606471525.89,357311.28,6.34
Big Spenders,3724,1121688914.83,301205.40,4.7
Recent Customers,3686,523618946.90,142056.14,2.94
Loyal Customers,3250,894919444.31,275359.83,5.64
Needs Attention,2295,312878395.60,136330.46,2.83
At Risk,1798,311293461.62,173133.18,4.08


In [0]:
%sql

DESCRIBE DETAIL ecommerce.gold.customer_segmentation;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,360056e8-b962-4f38-835e-d81e6ea61cf0,ecommerce.gold.customer_segmentation,null,abfss://gold@stcommercesrc.dfs.core.windows.net/customer_segmentation,2026-10-03T17:16:39.288Z,2026-10-03T17:16:40Z,List(),List(),1,951310,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce.control
COMMENT 'Control and audit metadata for e-commerce data pipelines';

In [0]:
%sql
SHOW SCHEMAS IN ecommerce;

databaseName
bronze
control
default
gold
information_schema
silver


In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce.control.pipeline_audit
(
    pipeline_name STRING,
    table_name STRING,
    run_id STRING,
    event_blob_path STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    previous_watermark TIMESTAMP,
    current_watermark TIMESTAMP,
    rows_read BIGINT,
    rows_written BIGINT,
    status STRING,
    error_message STRING,
    trigger_type STRING
)
USING DELTA
COMMENT 'Pipeline execution audit and control records for the e-commerce data platform';

In [0]:
%sql
DESCRIBE ecommerce.control.pipeline_audit;

col_name,data_type,comment
pipeline_name,string,null
table_name,string,null
run_id,string,null
event_blob_path,string,null
start_time,timestamp,null
end_time,timestamp,null
previous_watermark,timestamp,null
current_watermark,timestamp,null
rows_read,bigint,null
rows_written,bigint,null


In [0]:
%sql
SELECT * 
FROM ecommerce.control.pipeline_audit;

pipeline_name,table_name,run_id,event_blob_path,start_time,end_time,previous_watermark,current_watermark,rows_read,rows_written,status,error_message,trigger_type
